# Parallelism in PyTorch: Core Ideas

This notebook focuses on the main question behind parallel training:

**What exactly are we splitting?**

- **Data parallelism** splits the **batch**.
- **Model parallelism** splits the **model**.
- **Pipeline and tensor parallelism** are more structured forms of model parallelism.

The goal here is not to build a production training stack. The goal is to show the mechanism as clearly as possible with small PyTorch examples.

## Quick map

| Method | Split target | Best when | Main cost |
| --- | --- | --- | --- |
| Single process | Nothing | Model fits on one device | No parallel speedup |
| Data parallel (`DDP`) | Batch | Model fits on each GPU | Gradient synchronization |
| Model parallel | Layers or tensors | Model does not fit on one GPU | Activation transfer / collectives |
| Pipeline parallel | Depth + microbatches | Deep models across multiple GPUs | Pipeline bubbles |
| Tensor parallel | Weight matrices / attention heads | Very large layers | In-layer communication |

In [ ]:
import copy
import time

import torch
import torch.distributed as dist
import torch.nn as nn
import torch.optim as optim
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import DataLoader, DistributedSampler, TensorDataset


torch.manual_seed(0)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(0)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

primary_device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
num_devices = torch.cuda.device_count()
input_dim, hidden_dim, output_dim = 256, 512, 10
batch_size = 1024

features = torch.randn(8192, input_dim)
labels = torch.randint(0, output_dim, (8192,))
dataset = TensorDataset(features, labels)


class TinyMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim),
        )

    def forward(self, x):
        return self.net(x)


def step_once(model, batch, optimizer, loss_fn, device):
    x, y = batch
    x = x.to(device)
    y = y.to(device)
    optimizer.zero_grad(set_to_none=True)
    logits = model(x)
    loss = loss_fn(logits, y)
    loss.backward()
    optimizer.step()
    return loss.item()


print({
    "primary_device": str(primary_device),
    "cuda_device_count": num_devices,
    "dataset_size": len(dataset),
    "batch_size": batch_size,
})

## 1. Minimal Single-Process Baseline

A baseline matters because every parallel method adds overhead.

The single-process workflow is the reference path:

1. move the batch to one device
2. run forward
3. compute loss
4. run backward
5. update parameters

If this baseline is already under-utilizing the device, parallelism will not fix the real bottleneck.

In [ ]:
def benchmark_single_process(steps=20, device=primary_device):
    model = TinyMLP().to(device)
    optimizer = optim.AdamW(model.parameters(), lr=1e-3)
    loss_fn = nn.CrossEntropyLoss()
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, drop_last=True)

    iterator = iter(loader)
    start = time.perf_counter()
    last_loss = None
    for _ in range(steps):
        try:
            batch = next(iterator)
        except StopIteration:
            iterator = iter(loader)
            batch = next(iterator)
        last_loss = step_once(model, batch, optimizer, loss_fn, device)
    elapsed = time.perf_counter() - start
    samples = steps * batch_size
    return {
        "mode": "single_process",
        "device": str(device),
        "ms_per_step": round(1000 * elapsed / steps, 2),
        "samples_per_sec": round(samples / elapsed, 2),
        "last_loss": round(last_loss, 4),
    }


baseline_metrics = benchmark_single_process()
baseline_metrics

## 2. Data Parallelism with `DistributedDataParallel`

The idea is simple:

- every process keeps a full copy of the model
- each process sees a different shard of the batch
- gradients are synchronized during backward

So the **model is replicated**, but the **data is partitioned**.

This is the default choice when the model fits on each GPU and you want better throughput.

**Important notebook caveat:** real DDP is usually launched with `torchrun`, not by pressing Run Cell in a normal notebook session. The code below is the minimal worker logic.

In [ ]:
def ddp_worker(rank, world_size, steps=20, backend="nccl"):
    dist.init_process_group(backend=backend, rank=rank, world_size=world_size)
    torch.cuda.set_device(rank)
    device = torch.device(f"cuda:{rank}")

    model = TinyMLP().to(device)
    model = DDP(model, device_ids=[rank])
    optimizer = optim.AdamW(model.parameters(), lr=1e-3)
    loss_fn = nn.CrossEntropyLoss()

    sampler = DistributedSampler(
        dataset,
        num_replicas=world_size,
        rank=rank,
        shuffle=True,
    )
    loader = DataLoader(
        dataset,
        batch_size=batch_size // world_size,
        sampler=sampler,
        drop_last=True,
    )

    iterator = iter(loader)
    start = time.perf_counter()
    for step in range(steps):
        sampler.set_epoch(step)
        try:
            batch = next(iterator)
        except StopIteration:
            iterator = iter(loader)
            batch = next(iterator)
        step_once(model, batch, optimizer, loss_fn, device)
    elapsed = time.perf_counter() - start

    if rank == 0:
        print({
            "mode": "ddp",
            "world_size": world_size,
            "global_batch_size": batch_size,
            "ms_per_step": round(1000 * elapsed / steps, 2),
            "samples_per_sec": round((steps * batch_size) / elapsed, 2),
        })

    dist.destroy_process_group()


print("Launch idea: torchrun --nproc_per_node=2 your_script.py")

## 3. Simple Model Parallelism Across Devices

Now we split the model itself.

A small two-stage example captures the core mechanism:

- stage 1 lives on one device
- stage 2 lives on another device
- the activation tensor is moved between them during forward

This is the basic building block behind more advanced schemes:

- **pipeline parallelism** splits by depth and overlaps microbatches
- **tensor parallelism** splits one large layer across devices instead of splitting by stage

So model parallelism answers a different problem from DDP: it helps when one full model copy does **not** fit comfortably on one device.

In [ ]:
class TwoStageMLP(nn.Module):
    def __init__(self, device0, device1):
        super().__init__()
        self.device0 = torch.device(device0)
        self.device1 = torch.device(device1)
        self.stage0 = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
        ).to(self.device0)
        self.stage1 = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim),
        ).to(self.device1)

    def forward(self, x):
        x = x.to(self.device0)
        hidden = self.stage0(x)
        hidden = hidden.to(self.device1)
        return self.stage1(hidden)


def benchmark_model_parallel(steps=20):
    if num_devices < 2:
        return {"mode": "model_parallel", "status": "skipped: need at least 2 CUDA devices"}

    model = TwoStageMLP("cuda:0", "cuda:1")
    optimizer = optim.AdamW(model.parameters(), lr=1e-3)
    loss_fn = nn.CrossEntropyLoss()
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, drop_last=True)

    iterator = iter(loader)
    start = time.perf_counter()
    last_loss = None
    for _ in range(steps):
        try:
            batch = next(iterator)
        except StopIteration:
            iterator = iter(loader)
            batch = next(iterator)

        x, y = batch
        optimizer.zero_grad(set_to_none=True)
        logits = model(x)
        loss = loss_fn(logits, y.to(model.device1))
        loss.backward()
        optimizer.step()
        last_loss = loss.item()

    elapsed = time.perf_counter() - start
    return {
        "mode": "model_parallel",
        "devices": ["cuda:0", "cuda:1"],
        "ms_per_step": round(1000 * elapsed / steps, 2),
        "samples_per_sec": round((steps * batch_size) / elapsed, 2),
        "last_loss": round(last_loss, 4),
    }


model_parallel_metrics = benchmark_model_parallel()
model_parallel_metrics

## 4. Throughput and Trade-Offs

The benchmark numbers should be read as **mechanism demos**, not as universal conclusions.

The important interpretation is:

- **Single process** has the least orchestration overhead.
- **DDP** often gives the cleanest throughput scaling when the model fits per GPU.
- **Model parallel** solves memory pressure, but activation movement can reduce speedup.
- **Pipeline parallel** improves device utilization by feeding microbatches through stages.
- **Tensor parallel** is useful when a single layer is too large, but it introduces collectives inside the layer.

A compact decision rule:

| Situation | Prefer |
| --- | --- |
| Model fits on one GPU, need faster training | `DistributedDataParallel` |
| Model does not fit on one GPU | Model parallel / FSDP / tensor parallel |
| Very deep network across multiple GPUs | Pipeline parallel |
| Very large attention/MLP layers | Tensor parallel |
| Large-scale training | Combine DDP with a model-parallel strategy |

In practice, modern large-model training is often **hybrid parallelism**:

$$
\text{global parallelism} = \text{data parallel} + \text{model parallel} + \text{pipeline scheduling}
$$

The key mental model is still simple:

- split **data** when memory is fine and you want throughput
- split **model** when memory is the real bottleneck
- combine both when scaling to large models and large clusters

In [ ]:
comparison = [baseline_metrics, model_parallel_metrics]
for item in comparison:
    print(item)

print("DDP benchmark note: run ddp_worker(rank, world_size) from a torchrun-launched script.")